# B6: Token visibility, cost tracking and chargeback
1,000 calls from 3 team API keys (500/300/200). Platform-side numbers come from Prometheus (Limitador `authorized_hits` = tokens,
labelled by subscription / user / cost centre / model). They are compared with a client-side tally and priced at IDR 0.25/token.

In [ ]:
import os, sys, json
os.chdir(os.path.expanduser("~/aigov"))           # notebooks/ and data/ live here
sys.path.insert(0, "notebooks")
import harness as h
print("Guardrails:", os.environ["NEMO_URL"])

In [ ]:
import b6_chargeback
rows = b6_chargeback.main()

**Reading the output:** the script sends the 1,000 calls, waits about 75 s for Prometheus to scrape the gateway counters, then prints:

* `window … UTC, price IDR 0.25/token -> reports/chargeback-….csv`: the measurement window and where the CSV was saved.
* a table with one row per team (explained under the next cell).
* `TOTAL platform N tokens (IDR …) vs client M: diff 0.000%`: the gateway's count (what you would charge) versus what the applications counted themselves from each response. **0.000%** means they agree exactly.

In [ ]:
import pandas as pd
pd.DataFrame(rows)

**Columns:**

| Column | Meaning |
|---|---|
| `team_subscription` | the team's MaaS subscription (one per application team) |
| `cost_center` | the cost centre the subscription is labelled with: who gets charged |
| `model` | the model used |
| `platform_calls` / `platform_tokens` | requests and tokens counted by the gateway in the window |
| `cost_idr` | `platform_tokens × price` (IDR 0.25 per token here) |
| `client_calls_ok` / `client_calls_failed` | what the client saw: successful and failed calls |
| `client_tokens` | tokens the client summed from each response's `usage.total_tokens` |
| `difference_pct` | `(platform − client) / client × 100`. 0.0 = exact match |

A non-zero difference usually means the window caught other traffic from the same team, or failed calls; check `client_calls_failed`.